In [ ]:
import base64
import re
import time
import uuid

import requests
import jwt
from cryptography.hazmat.primitives.asymmetric import ec

so the idea is that we include our public key as part of the token, that's what the `alg` and `jwk` are for. my understanding is that the JWK (JSON Web Key) basically contains all the relevant information for whatever encryption algorithm you're using. mercari uses elliptic curve so we too use elliptic curve, and so in `headers` we have `alg` set to `"ES256"`, and then in JWK, where we specify all the things we need to for our particular encryption method, we say "here is the x and y", as the x/y is all a public key is for elliptic curve. ok!

also, base64url gets used instead of base64 because base64 will include '+' and '\' characters, which will not play nice with the fact that this is being sent in an HTTP header. 

i'm not sure how much of this process was truly necessary, since on mercari you view results without being logged in, and hence this token has no real meaning. but at the same time, if you try to make a GET/POST without including a valid one in `DPoP` of the header, you get 401'd.

In [ ]:
# generate a key pair once and reuse it across requests
PRIVATE_KEY = ec.generate_private_key(ec.SECP256R1())
PUBLIC_KEY = PRIVATE_KEY.public_key()

def int_to_base64url(n:int, length:int=32) -> str:
    # all decode does is convert this from bytes to a utf-8 string
    # even though this is base64url the python function leaves in the padding '=' chars so remove them
    # length = 32 bytes because we're using, well, a 256 bit key
    return base64.urlsafe_b64encode(n.to_bytes(length, byteorder="big")).decode().rstrip("=")

# thank you https://www.jwt.io/
JWK = {
    "crv": "P-256",
    "kty": "EC",
    "x": int_to_base64url(PUBLIC_KEY.public_numbers().x),
    "y": int_to_base64url(PUBLIC_KEY.public_numbers().y),
}

# the only thing that actually needs to change between requests is the payload
def generate_dpop_proof(method: str, url: str) -> str:
    headers = {
        "typ": "dpop+jwt",
        "alg": "ES256",
        "jwk": JWK,
    }
    payload = {
        "jti": str(uuid.uuid4()),   # Unique ID per request
        "htm": method,              # HTTP method
        "htu": url,                  # Target URL
        "iat": int(time.time()),     # Current timestamp
    }
    # this does the "signature" part of the header.payload.signature format
    return jwt.encode(
        payload,
        PRIVATE_KEY,
        algorithm="ES256",
        headers=headers
    )

In [ ]:
# this is here half because i keep forgetting to update the header information,
# and half because it was annoying to keep mutating this global dict every time i make a request
def make_valid_header(http_method:str, url:str) -> dict:
    MERCARI_HEADERS = {
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:141.0) Gecko/20100101 Firefox/141.0',
        'Accept': 'application/json, text/plain, */*',
        'Accept-Language': 'ja',
        'Accept-Encoding': 'gzip, deflate, br, zstd',
        'Content-Type': 'application/json',
        'X-Platform': 'web',
        'X-Country-Code': 'US',
        'Origin': 'https://jp.mercari.com',
        'Connection': 'keep-alive',
        'Referer': 'https://jp.mercari.com/',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'cross-site',
        'Idempotency-Key': '"11432578277613167492"',
        'Pragma': 'no-cache',
        'Cache-Control': 'no-cache',
        'TE': 'trailers',
    }

    ret = dict(MERCARI_HEADERS)
    ret.update({'DPoP': generate_dpop_proof(http_method, url)})
    return ret

In [ ]:
# get the conversion rate since shops pages just don't send USD prices
CURRENCY_URL = 'https://api.mercari.jp/v2/getCurrencyConversionRate/currency'
jpy_to_usd_response = requests.get(CURRENCY_URL, params={'currency_code' : 'USD'}, headers=make_valid_header("GET", CURRENCY_URL))
print(jpy_to_usd_response)
JPY_TO_USD = jpy_to_usd_response.json()['rate']
print(JPY_TO_USD)

In [ ]:
# get the ids of the 120 (or potentially less) results that would appear doing a normal search for the given term.
# always sorts by newest.
def get_ids_from_search(search:str):
    JSON_DATA = {
        'userId': '',
        'config': {
            'responseToggles': [
                'QUERY_SUGGESTION_WEB_1',
            ],
        },
        'pageSize': 120,
        'pageToken': '',
        'searchSessionId': 'c83bbb3d087c6baa398a460d06a0cefc',
        'source': 'BaseSerp',
        'indexRouting': 'INDEX_ROUTING_UNSPECIFIED',
        'thumbnailTypes': [],
        'searchCondition': {
            'keyword': search,
            'excludeKeyword': '',
            'sort': 'SORT_CREATED_TIME',
            'order': 'ORDER_DESC',
            'status': [],
            'sizeId': [],
            'categoryId': [],
            'brandId': [],
            'sellerId': [],
            'priceMin': 0,
            'priceMax': 0,
            'itemConditionId': [],
            'shippingPayerId': [],
            'shippingFromArea': [],
            'shippingMethod': [],
            'colorId': [],
            'hasCoupon': False,
            'attributes': [],
            'itemTypes': [],
            'skuIds': [],
            'shopIds': [],
            'excludeShippingMethodIds': [],
        },
        'serviceFrom': 'suruga',
        'withItemBrand': True,
        'withItemSize': False,
        'withItemPromotions': True,
        'withItemSizes': True,
        'withShopname': False,
        'useDynamicAttribute': True,
        'withSuggestedItems': True,
        'withOfferPricePromotion': True,
        'withProductSuggest': True,
        'withParentProducts': False,
        'withProductArticles': True,
        'withSearchConditionId': False,
        'withAuction': True,
        'laplaceDeviceUuid': 'fa3a3ca3-e696-4d01-a2f5-49eb502e1c38',
    }
    URL = "https://api.mercari.jp/v2/entities:search"
    response = requests.post(URL, json=JSON_DATA, headers=make_valid_header("POST", URL))

    if not response.ok:
        print("Search response:", response.status_code)
        return None

    json = response.json()
    ret = []
    for item in json['items']:
        ret.append(item['id'])
    return ret

In [ ]:
def make_listing_from_id(id:str):
    headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:141.0) Gecko/20100101 Firefox/141.0',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'ja',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Content-Type': 'application/json',
    'X-Platform': 'web',
    'X-Country-Code': 'US',
    'Origin': 'https://jp.mercari.com',
    'Connection': 'keep-alive',
    'Referer': 'https://jp.mercari.com/',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'cross-site',
    'Idempotency-Key': '"11432578277613167492"', # this doesn't actually seem necessary?
    'Pragma': 'no-cache',
    'Cache-Control': 'no-cache',
    'TE': 'trailers',
    }
    item_pattern = re.compile("m[0-9]{9,13}")
    if item_pattern.match(id):
        # i don't know what their engineers are cooking up here
        url = "https://api.mercari.jp/items/get?id={0}&include_item_attributes=true&include_product_page_component=true&include_non_ui_item_attributes=true&include_donation=true&include_item_attributes_sections=true&include_auction=true".format(id)
        item_type = "user"
    else:
        # i guess 'shops' is a newer feature so they got the tech-debtless API
        url = "https://api.mercari.jp/v1/marketplaces/shops/products/{0}?view=FULL".format(id)
        item_type = "shop"

    headers.update({'DPoP': generate_dpop_proof("GET", url)})
    response = requests.get(url, headers=headers)
    print(response)
    if not response.ok:
        return None

    jason = response.json()
    # the two apis have fairly different JSON structures? ok
    if item_type == "user":
        return {
            "id": id,
            "title": jason['data']['name'],
            "description" : jason['data']['description'],
            "price_usd" : round((float(jason['data']['price']) * JPY_TO_USD), 2),
            "url" : "https://buyee.jp/mercari/item/" + id
        }
    else:
        return {
            "id": id,
            "title": jason['displayName'],
            "description" : jason['productDetail']['description'],
            "price_usd" : round((float(jason['price']) * JPY_TO_USD), 2),
            "url" : "https://buyee.jp/mercari/item/" + id
        }

In [ ]:
listing_ids = get_ids_from_search("professional2")

# obviously TODO here for when this is more, like, a real program.
# probably retry a few times with some delay? definitely log it somewhere
if listing_ids is None:
    raise Exception("I am kill")

import time
listings = []
for idx in range(10):
    tmp = make_listing_from_id(listing_ids[idx])
    time.sleep(0.5)
    if tmp != None:
        listings.append(tmp)
    else:
        continue

In [ ]:
for x in listings:
    print(x['title'], "(${0})".format(x['price_usd']))

I'm going to figure this part out later since I'm not sure exactly how I want to approach it.

In [ ]:
# just take this to be some black box that will take in a listing in the form created above,
# along with a target that's like target = { title = <str>, description = <str> }
def is_listing_relevant(listing:dict, target:dict) -> bool:
    return True

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
# specifically using environ() rather than getenv() so it doesn't complain about how the variable might not be present
NTFY_URL = os.environ['NTFY_URL']
NTFY_TOKEN = os.environ["NTFY_TOKEN"]

So, `data` is the body of the message, and then to specify anything else you need to use the `headers`.

In [ ]:
test_id = "m66339163678"
requests.post(NTFY_URL,
data=f"""
# header 1
content under my sweet header
## header 2
some more sweet content. a [link to a buyee listing](https://buyee.jp/mercari/item/{test_id})
""",
headers={
    "Authorization": f"Bearer {NTFY_TOKEN}",
    "Title": "they're gonna get you",
    "Markdown": "yes"
})